# Chronos-2 with covariates

parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [1]:
!pip install 'chronos-forecasting[extras]>=2.2' pyarrow pandas scikit-learn -q
print("Restart kernel before running script further")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=

## Config

In [2]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from chronos import BaseChronosPipeline, Chronos2Pipeline
try:
    from IPython.display import display
except ImportError:
    pass


PARQUET_PATH = os.environ.get(
    "EEG_PARQUET", os.path.join(os.getcwd(), "eeg_preprocessed_4ch_raw.parquet")
)
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = DATA_PATH

LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

COVARIATE_MAP = {
    "Fp1": "P3",
    "P3": "Fp1",
    "Fp2": "P4",
    "P4": "Fp2"
}

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

2026-06-05 20:49:19.485762: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780692559.672276      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780692559.726879      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780692560.184009      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780692560.184060      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780692560.184063      23 computation_placer.cc:177] computation placer alr

Loaded parquet: /kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet | rows in file: 88, evaluation on (the first) 88 subjects.


## Model

In [3]:
CSV_OUT = "chronos2_eeg_cov_eval_results.csv"
REPO_ID = "amazon/chronos-2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Chronos-2 ({REPO_ID}), device: {device}...")

model = BaseChronosPipeline.from_pretrained(
    REPO_ID, 
    device_map=device,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
)
print("Chronos-2 ready.")

def forecast_windows(model, y_target, y_cov, target_ch_name, cov_ch_name, starts, ctx_len, hor_len):
    preds_all = []
    base_date = pd.Timestamp("2024-01-01")
    
    for s in starts:
        ctx_target = y_target[s : s + ctx_len]
        ctx_cov = y_cov[s : s + ctx_len]
        fut_cov = y_cov[s + ctx_len : s + ctx_len + hor_len]

        ctx_dates = [base_date + pd.Timedelta(seconds=x) for x in range(ctx_len)]
        fut_dates = [base_date + pd.Timedelta(seconds=x) for x in range(ctx_len, ctx_len + hor_len)]

        df_ctx = pd.DataFrame({
            "item_id": ["EEG"] * ctx_len,
            "timestamp": ctx_dates, 
            target_ch_name: ctx_target,  
            cov_ch_name: ctx_cov  
        })

        df_fut = pd.DataFrame({
            "item_id": ["EEG"] * hor_len,
            "timestamp": fut_dates,             
            cov_ch_name: fut_cov  
        })

        pred_df = model.predict_df(
            df=df_ctx,
            future_df=df_fut,                   
            prediction_length=hor_len,
            id_column="item_id",
            timestamp_column="timestamp",
            target=target_ch_name,
            quantile_levels=[0.5]
        )
        
        preds = pred_df["predictions"].values
        preds_all.append(preds)
        
    return np.array(preds_all)

Loading Chronos-2 (amazon/chronos-2), device: cuda...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

Chronos-2 ready.


## Processing functions

In [4]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

## Evaluation loop

In [5]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    
    for ch in CHANNELS:
        cov_ch = COVARIATE_MAP[ch]
        y_target = series_to_numpy(row[ch])
        y_cov = series_to_numpy(row[cov_ch])
        
        starts = window_starts(len(y_target), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
            
        # --- Unified forecast call ---
        preds = forecast_windows(
            model=model, 
            y_target=y_target, 
            y_cov=y_cov, 
            target_ch_name=ch, 
            cov_ch_name=cov_ch, 
            starts=starts, 
            ctx_len=CONTEXT_LENGTH, 
            hor_len=HORIZON_LENGTH
        )
        
        mses_win = []
        for i, s in enumerate(starts):
            tgt_true = y_target[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt_true, preds[i]))
            
        mse_mean = float(np.mean(mses_win))
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "covariate_used": cov_ch,
                "mse": mse_mean,
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "covariate_used": "ALL",
            "mse": float(sub["mse"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "covariate_used": COVARIATE_MAP[ch],
                "mse": float(sub_ch["mse"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out.tail(20)) # display end of csv
except ImportError:
    print(df_out.tail(20).to_string())

Subjects analysed: 1 / 88 (last: sub-001).
Subjects analysed: 2 / 88 (last: sub-002).
Subjects analysed: 3 / 88 (last: sub-003).
Subjects analysed: 4 / 88 (last: sub-004).
Subjects analysed: 5 / 88 (last: sub-005).
Subjects analysed: 6 / 88 (last: sub-006).
Subjects analysed: 7 / 88 (last: sub-007).
Subjects analysed: 8 / 88 (last: sub-008).
Subjects analysed: 9 / 88 (last: sub-009).
Subjects analysed: 10 / 88 (last: sub-010).
Subjects analysed: 11 / 88 (last: sub-011).
Subjects analysed: 12 / 88 (last: sub-012).
Subjects analysed: 13 / 88 (last: sub-013).
Subjects analysed: 14 / 88 (last: sub-014).
Subjects analysed: 15 / 88 (last: sub-015).
Subjects analysed: 16 / 88 (last: sub-016).
Subjects analysed: 17 / 88 (last: sub-017).
Subjects analysed: 18 / 88 (last: sub-018).
Subjects analysed: 19 / 88 (last: sub-019).
Subjects analysed: 20 / 88 (last: sub-020).
Subjects analysed: 21 / 88 (last: sub-021).
Subjects analysed: 22 / 88 (last: sub-022).
Subjects analysed: 23 / 88 (last: sub-023

,record_type,subject_id,group,electrode,covariate_used,mse,n_windows
347,per_patient_electrode,sub-087,F,P4,Fp2,1.191020e-10,5
348,per_patient_electrode,sub-088,F,Fp1,P3,2.203015e-10,5
349,per_patient_electrode,sub-088,F,Fp2,P4,1.892454e-10,5
350,per_patient_electrode,sub-088,F,P3,Fp1,2.810585e-10,5
351,per_patient_electrode,sub-088,F,P4,Fp2,1.967802e-10,5
352,group_all_electrodes,,A,ALL,ALL,2.268108e-10,5
353,group_per_electrode,,A,Fp1,P3,2.139465e-10,5
354,group_per_electrode,,A,Fp2,P4,2.560174e-10,5
355,group_per_electrode,,A,P3,Fp1,2.226637e-10,5
356,group_per_electrode,,A,P4,Fp2,2.146157e-10,5
